# Preprocessing: Sea-Ice Concentration

**Manuscript reference:** Sect. 2.2 (satellite data), Sect. 4 (Results — spatial context in Fig. 3)

Prepares the daily MODIS/AMSR-2 merged sea-ice concentration (SIC) product used as surface context in Fig. 3 (flux map overview). The merged product combines MODIS visible imagery and AMSR-2 passive-microwave retrieval. Daily SIC maps are regridded to 1 km resolution and stored in `satellite_unified/sic_modis-aqua_amsr2-gcom-w1_merged_nh_1000m_APRIL.nc`.


In [ ]:
import os 
import numpy as np
import rasterio
import xarray as xr 
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import rioxarray
import pandas as pd
import glob
from tqdm import tqdm

path = '../../../data/observations/raw/satellite/modis/modis_amsr2_sic'

# walk through the directory and subdirectories and get all the files 

files = glob.glob(os.path.join(path, '**', '*.tif'), recursive=True)  

list_of_datasets = []

files.sort()


for file in tqdm(files):

    datestr = file.split('/')[-1].split('_')[-1].split('.')[0]

    datetime = pd.to_datetime(datestr, format='%Y%m%d')

    ds = xr.open_dataset(file, engine='rasterio')
    xds_lonlat = ds.rio.reproject("EPSG:4326").sel(x=slice(-60, 60), y=slice(90, 60))
    xds_lonlat = xds_lonlat.where(xds_lonlat < 150)
    xds_lonlat['time'] = datetime
    list_of_datasets.append(xds_lonlat)

xds = xr.concat(list_of_datasets, dim='time')
xds = xds.rename({'x': 'lon', 'y': 'lat', 'z': 'sic'})
xds['sic'].attrs['long_name'] = 'Sea Ice Concentration'
xds = xds.sortby('time')
xds.to_netcdf('../../../data/observations/processed/satellite_unified/sic_modis-aqua_amsr2-gcom-w1_merged_nh_1000m_APRIL_large.nc', mode='w')


In [ ]:
from cartopy import crs as ccrs
from cartopy import feature as cfeature

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.NorthPolarStereo()})
ax.coastlines()
xds.sic.mean('time').plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', vmin=0, vmax=100) 
ax.set_extent([-45, 45, 60, 90], ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, zorder=0, facecolor='lightgray')

plt.title('Mean Sea Ice Concentration April MODIS-AMSR2')


In [ ]:
xds.sic.where(xds.sic <= 100).mean(['lat', 'lon']).plot()